# SARIMAX: 72-hour conductivity forecasting with exogenous variables

This notebook adds a leakage-aware `statsmodels.tsa.statespace.SARIMAX` experiment to the existing paper workflow. It forecasts **Indusii conductivity for the next 72 hours at 15-minute resolution** from:

- the target's own autoregressive history;
- upstream conductivity at configurable propagation-lag hypotheses;
- available environmental variables;
- compact Fourier terms for weekly and annual cycles; and
- a daily SARIMAX seasonal period of `m=96`.

The chronological contract matches the other comparison notebooks:

- **January–July 2025:** parameter estimation;
- **August 2025:** SARIMAX-order selection, Terneuzen-lag selection, and no test inspection;
- **September 2025:** held-out evaluation only.

The existing 42-hour Terneuzen shift is the baseline hypothesis, not an assumed physical truth. After order selection, a small validation-only sensitivity step compares 36, 42, and 48 hours. The full requested order grid is defined, but the default run uses a curated 12-candidate subset because the Cartesian grid contains 256 seasonal state-space fits and can be extremely expensive at 15-minute resolution.

No cells in this notebook were executed when it was created.


In [ ]:
from pathlib import Path
from itertools import product
import gc, hashlib, json, os, sys, time, warnings

ROOT = Path.cwd().resolve()
if not (ROOT / "lag_analytics_workspace").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".matplotlib-cache"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tsa.statespace.sarimax import SARIMAX

from time_series_analysis.paper_experiment_utils import (
    EVALUATION_HOURS,
    FORECAST_HOURS,
    FORECAST_STEPS,
    RESAMPLE_MINUTES,
    SENSOR_NAMES,
    SPLIT_PERIODS,
    STEPS_PER_HOUR,
    TARGET,
    load_prepared_observations,
)

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Search and evaluation controls. "recommended" is the default reproducible run.
SEARCH_PROFILE = "recommended"  # "quick", "recommended", or "full"
MAXITER = 75
VALIDATION_ORIGIN_STRIDE_STEPS = 6 * STEPS_PER_HOUR  # one origin every 6 hours
TEST_ORIGIN_STRIDE_STEPS = STEPS_PER_HOUR            # one origin every hour
INTERVAL_ALPHA = 0.10                                # central 90% interval
FORCE_REFIT = False
FUTURE_EXOG_POLICY = "causal_persistence"  # or "observed_hindcast" (diagnostic only)

# Propagation lags are hypotheses. Only August may choose among Terneuzen values.
TERNEUZEN_LAG_HYPOTHESES = (36, 42, 48)
BASELINE_TERNEUZEN_LAG_HOURS = 42
OTHER_STATION_LAG_HOURS = {
    "Westdorpe": 30,
    "Gent - far": 18,
    "Gent - near": 6,
}

WEEKLY_FOURIER_HARMONICS = 2
ANNUAL_FOURIER_HARMONICS = 2
MAX_CAUSAL_FILL_STEPS = 8  # at most two hours of forward filling

# Future values of these columns may be used exactly only if they are genuinely
# scheduled or forecast before the forecast origin. Empty is the safe default.
KNOWN_FUTURE_COLUMNS = set()

OPTIONAL_ENVIRONMENT_FILE = ROOT / "data" / "sarimax_environmental.csv"
OPTIONAL_ENVIRONMENT_COLUMNS = (
    "freshwater_discharge",
    "temperature",
    "lock_operations",
    "vessel_activity",
)

CACHE_DIR = ROOT / "time_series_analysis" / "model_cache" / "sarimax"
RESULTS_DIR = ROOT / "time_series_analysis" / "results" / "sarimax"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("statsmodels", statsmodels.__version__)
print("Forecast steps:", FORECAST_STEPS, "at", RESAMPLE_MINUTES, "minutes")


## 1. Load the matched conductivity data and available environmental series

The five conductivity series use the same Fuseki loader, unit normalization, 15-minute grid, short-gap treatment, and causal trailing two-hour smoothing as the other paper notebooks.

The repository currently supplies:

- `data/waterlevel.csv`: water level, used here as a **tide proxy**;
- `data/ertveldeprecipitation.csv`: Ertvelde rainfall.

The current water-level file has no populated runoff/discharge field, and the repository has no aligned lock-operation or vessel-activity series. If available later, place a file at `data/sarimax_environmental.csv` with a `timestamp` column and any of:

`freshwater_discharge, temperature, lock_operations, vessel_activity`

Only present numeric columns are included. The audit table below makes omissions explicit.


In [ ]:
def _series_from_csv(path, timestamp_column, value_column, name, *, sep=",", decimal=".", aggregation="mean"):
    frame = pd.read_csv(path, sep=sep, decimal=decimal, low_memory=False)
    if timestamp_column not in frame or value_column not in frame:
        raise ValueError(f"{path.name} must contain {timestamp_column!r} and {value_column!r}.")
    timestamps = pd.to_datetime(frame[timestamp_column], utc=True, errors="coerce")
    values = pd.to_numeric(frame[value_column], errors="coerce")
    series = pd.Series(values.to_numpy(), index=timestamps, name=name).dropna()
    series = series.groupby(level=0).mean().sort_index()
    resampler = series.resample(f"{RESAMPLE_MINUTES}min")
    if aggregation == "sum":
        return resampler.sum(min_count=1).rename(name)
    return resampler.mean().rename(name)


def load_environmental_series(index):
    series = {}
    source_rows = []

    waterlevel_path = ROOT / "data" / "waterlevel.csv"
    if waterlevel_path.exists():
        waterlevel_frame = pd.read_csv(waterlevel_path, low_memory=False)
        timestamp_column = "Timestamp"
        timestamps = pd.to_datetime(waterlevel_frame[timestamp_column], utc=True, errors="coerce")
        for output_name, candidates in {
            "water_level_tide_proxy": ("Value", "Water Depth", "Stagesource Value"),
            "freshwater_discharge": ("Runoff Value", "Runoff"),
        }.items():
            source_column = next(
                (
                    candidate for candidate in candidates
                    if candidate in waterlevel_frame
                    and pd.to_numeric(waterlevel_frame[candidate], errors="coerce").notna().any()
                ),
                None,
            )
            if source_column is not None:
                values = pd.to_numeric(waterlevel_frame[source_column], errors="coerce")
                values = pd.Series(values.to_numpy(), index=timestamps, name=output_name).dropna()
                series[output_name] = values.groupby(level=0).mean().sort_index().resample(
                    f"{RESAMPLE_MINUTES}min"
                ).mean()
                source_rows.append({"Feature": output_name, "Source": str(waterlevel_path), "Status": f"loaded from {source_column}"})
            else:
                source_rows.append({"Feature": output_name, "Source": str(waterlevel_path), "Status": "not populated"})
    else:
        source_rows.append({"Feature": "water_level_tide_proxy", "Source": str(waterlevel_path), "Status": "file missing"})

    rainfall_path = ROOT / "data" / "ertveldeprecipitation.csv"
    if rainfall_path.exists():
        series["rainfall_15min"] = _series_from_csv(
            rainfall_path, "timestamp", "Value", "rainfall_15min",
            sep=";", decimal=",", aggregation="sum",
        )
        source_rows.append({"Feature": "rainfall_15min", "Source": str(rainfall_path), "Status": "loaded"})
    else:
        source_rows.append({"Feature": "rainfall_15min", "Source": str(rainfall_path), "Status": "file missing"})

    if OPTIONAL_ENVIRONMENT_FILE.exists():
        optional = pd.read_csv(OPTIONAL_ENVIRONMENT_FILE, low_memory=False)
        timestamp_name = next((c for c in optional.columns if c.lower() in {"timestamp", "time", "datetime"}), None)
        if timestamp_name is None:
            raise ValueError(f"{OPTIONAL_ENVIRONMENT_FILE} needs a timestamp column.")
        timestamps = pd.to_datetime(optional[timestamp_name], utc=True, errors="coerce")
        for name in OPTIONAL_ENVIRONMENT_COLUMNS:
            if name not in optional:
                source_rows.append({"Feature": name, "Source": str(OPTIONAL_ENVIRONMENT_FILE), "Status": "column missing"})
                continue
            values = pd.to_numeric(optional[name], errors="coerce")
            if not values.notna().any():
                source_rows.append({"Feature": name, "Source": str(OPTIONAL_ENVIRONMENT_FILE), "Status": "not populated"})
                continue
            raw = pd.Series(values.to_numpy(), index=timestamps, name=name).dropna().groupby(level=0).mean().sort_index()
            if name in {"lock_operations", "vessel_activity"}:
                series[name] = raw.resample(f"{RESAMPLE_MINUTES}min").sum(min_count=1)
            else:
                series[name] = raw.resample(f"{RESAMPLE_MINUTES}min").mean()
            source_rows.append({"Feature": name, "Source": str(OPTIONAL_ENVIRONMENT_FILE), "Status": "loaded"})
    else:
        for name in OPTIONAL_ENVIRONMENT_COLUMNS:
            source_rows.append({"Feature": name, "Source": str(OPTIONAL_ENVIRONMENT_FILE), "Status": "optional file missing"})

    aligned = pd.DataFrame(index=index)
    for name, values in series.items():
        aligned[name] = values.reindex(index)
    return aligned, pd.DataFrame(source_rows)


data, conductivity_coverage, unit_report = load_prepared_observations()
data = data.loc[SPLIT_PERIODS["train"][0] : SPLIT_PERIODS["test"][1]].asfreq(
    f"{RESAMPLE_MINUTES}min"
)
environment_raw, environment_sources = load_environmental_series(data.index)

print(f"Prepared conductivity grid: {len(data):,} rows from {data.index.min()} to {data.index.max()}")
display(conductivity_coverage)
display(environment_sources)


## 2. Build causal exogenous designs

The default station-lag hypotheses are 42 hours for Terneuzen, 30 hours for Westdorpe, 18 hours for Gent-far, and 6 hours for Gent-near. These are configurable hypotheses reflecting station ordering—not established travel times.

For a 72-hour recursive forecast, the complete future path of a 42-hour-shifted regressor is only known for the first 42 hours. The default `causal_persistence` policy therefore uses the already-observed portion and freezes each stochastic regressor after its known-ahead limit. Fourier terms are known exactly. Set `observed_hindcast` only for a clearly labelled oracle diagnostic; it uses realized future covariates and is not a deployable forecast.

Missing exogenous values receive at most two hours of causal forward filling, followed by a training-median fallback and a missingness indicator. Exogenous scaling statistics are learned from January–July only.


In [ ]:
def add_fourier_terms(frame, period_steps, harmonics, prefix):
    t = np.arange(len(frame), dtype=np.float64)
    for harmonic in range(1, harmonics + 1):
        angle = 2.0 * np.pi * harmonic * t / period_steps
        frame[f"{prefix}_sin_{harmonic}"] = np.sin(angle)
        frame[f"{prefix}_cos_{harmonic}"] = np.cos(angle)


def build_design(terneuzen_lag_hours):
    index = data.index
    raw = pd.DataFrame(index=index)
    lag_hours = {"Terneuzen": int(terneuzen_lag_hours), **OTHER_STATION_LAG_HOURS}
    known_ahead_steps = {}

    for station, hours in lag_hours.items():
        name = f"{station.lower().replace(' ', '_').replace('-', '_')}_lag_{hours}h"
        raw[name] = data[station].shift(hours * STEPS_PER_HOUR)
        known_ahead_steps[name] = hours * STEPS_PER_HOUR

    for name in environment_raw:
        raw[name] = environment_raw[name]
        known_ahead_steps[name] = 0

    add_fourier_terms(
        raw,
        period_steps=7 * 24 * STEPS_PER_HOUR,
        harmonics=WEEKLY_FOURIER_HARMONICS,
        prefix="weekly",
    )
    add_fourier_terms(
        raw,
        period_steps=365.2425 * 24 * STEPS_PER_HOUR,
        harmonics=ANNUAL_FOURIER_HARMONICS,
        prefix="annual",
    )
    for name in [c for c in raw if c.startswith(("weekly_", "annual_"))]:
        known_ahead_steps[name] = FORECAST_STEPS

    # Use one common start for all Terneuzen lag hypotheses.
    common_lag_steps = max(TERNEUZEN_LAG_HYPOTHESES) * STEPS_PER_HOUR
    common_start = index[common_lag_steps]
    raw = raw.loc[common_start:].copy()
    endog = data[TARGET].loc[raw.index].astype(np.float64)

    train_start, train_end = (pd.Timestamp(value, tz="UTC") for value in SPLIT_PERIODS["train"])
    train_mask = (raw.index >= train_start) & (raw.index <= train_end)
    missing_before_fill = raw.isna()
    filled = raw.ffill(limit=MAX_CAUSAL_FILL_STEPS)

    keep_columns = []
    dropped = []
    for name in list(filled.columns):
        median = filled.loc[train_mask, name].median()
        if not np.isfinite(median):
            dropped.append(name)
            continue
        keep_columns.append(name)
        filled[name] = filled[name].fillna(median)
        if missing_before_fill[name].any():
            indicator = f"{name}__missing"
            filled[indicator] = missing_before_fill[name].astype(np.float64)
            known_ahead_steps[indicator] = known_ahead_steps[name]

    filled = filled[keep_columns + [c for c in filled if c.endswith("__missing") and c.rsplit("__missing", 1)[0] in keep_columns]]
    means = filled.loc[train_mask].mean()
    scales = filled.loc[train_mask].std().replace(0.0, 1.0).fillna(1.0)
    exog = ((filled - means) / scales).astype(np.float64)
    assert np.isfinite(exog.to_numpy()).all()

    coverage_rows = []
    for name in raw:
        coverage_rows.append({
            "Feature": name,
            "Train coverage": float(raw.loc[train_mask, name].notna().mean()),
            "Overall coverage": float(raw[name].notna().mean()),
            "Included": name in keep_columns,
        })
    return {
        "endog": endog,
        "exog": exog,
        "known_ahead_steps": {name: known_ahead_steps[name] for name in exog},
        "means": means,
        "scales": scales,
        "coverage": pd.DataFrame(coverage_rows),
        "dropped": dropped,
        "terneuzen_lag_hours": int(terneuzen_lag_hours),
    }


baseline_design = build_design(BASELINE_TERNEUZEN_LAG_HOURS)
print("Exogenous columns:", list(baseline_design["exog"].columns))
print("Dropped because no training observations:", baseline_design["dropped"] or "none")
display(baseline_design["coverage"].round(3))


## 3. Predeclared SARIMAX search

The complete requested Cartesian grid is available under `SEARCH_PROFILE="full"`:

- `p, q ∈ {0,1,2,3}`
- `d ∈ {0,1}`
- `P, Q ∈ {0,1}`
- `D ∈ {0,1}`
- `m = 96`

That is **256 fits**, not a small computation for a 15-minute seasonal state-space model. The default curated grid contains 12 representative candidates from the same value ranges. `quick` uses four candidates. Selection minimizes rolling 72-hour August RMSE; AIC is reported only as a diagnostic and never sees September.


In [ ]:
P_VALUES = Q_VALUES = (0, 1, 2, 3)
D_VALUES = (0, 1)
SEASONAL_P_VALUES = SEASONAL_Q_VALUES = (0, 1)
SEASONAL_D_VALUES = (0, 1)
DAILY_SEASONAL_STEPS = 24 * STEPS_PER_HOUR  # 96

FULL_CANDIDATES = [
    ((p, d, q), (P, D, Q, DAILY_SEASONAL_STEPS))
    for p, d, q, P, D, Q in product(
        P_VALUES,
        D_VALUES,
        Q_VALUES,
        SEASONAL_P_VALUES,
        SEASONAL_D_VALUES,
        SEASONAL_Q_VALUES,
    )
]

RECOMMENDED_CANDIDATES = [
    ((0, 0, 0), (1, 0, 0, DAILY_SEASONAL_STEPS)),
    ((1, 0, 0), (1, 0, 0, DAILY_SEASONAL_STEPS)),
    ((0, 0, 1), (0, 0, 1, DAILY_SEASONAL_STEPS)),
    ((1, 0, 1), (1, 0, 0, DAILY_SEASONAL_STEPS)),
    ((2, 0, 1), (0, 0, 1, DAILY_SEASONAL_STEPS)),
    ((1, 0, 2), (1, 0, 1, DAILY_SEASONAL_STEPS)),
    ((3, 0, 1), (1, 0, 0, DAILY_SEASONAL_STEPS)),
    ((1, 0, 3), (0, 0, 1, DAILY_SEASONAL_STEPS)),
    ((1, 1, 1), (0, 0, 0, DAILY_SEASONAL_STEPS)),
    ((2, 1, 1), (1, 0, 0, DAILY_SEASONAL_STEPS)),
    ((1, 1, 2), (0, 1, 1, DAILY_SEASONAL_STEPS)),
    ((3, 1, 0), (1, 0, 1, DAILY_SEASONAL_STEPS)),
]
QUICK_CANDIDATES = RECOMMENDED_CANDIDATES[:4]

if SEARCH_PROFILE == "quick":
    CANDIDATES = QUICK_CANDIDATES
elif SEARCH_PROFILE == "recommended":
    CANDIDATES = RECOMMENDED_CANDIDATES
elif SEARCH_PROFILE == "full":
    CANDIDATES = FULL_CANDIDATES
else:
    raise ValueError("SEARCH_PROFILE must be 'quick', 'recommended', or 'full'.")

candidate_table = pd.DataFrame(
    [{"Candidate": i, "order": order, "seasonal_order": seasonal} for i, (order, seasonal) in enumerate(CANDIDATES)]
)
print(f"Search profile {SEARCH_PROFILE!r}: {len(CANDIDATES)} of {len(FULL_CANDIDATES)} possible fits")
display(candidate_table)


In [ ]:
def period_mask(index, split_name):
    start, end = (pd.Timestamp(value, tz="UTC") for value in SPLIT_PERIODS[split_name])
    return (index >= start) & (index <= end)


def eligible_origin_positions(endog, split_name, stride_steps):
    index = endog.index
    start, end = (pd.Timestamp(value, tz="UTC") for value in SPLIT_PERIODS[split_name])
    positions = []
    for position in range(1, len(index) - FORECAST_STEPS + 1):
        if index[position] < start or index[position + FORECAST_STEPS - 1] > end:
            continue
        if (position - positions[0]) % stride_steps != 0 if positions else False:
            continue
        actual = endog.iloc[position : position + FORECAST_STEPS].to_numpy()
        if not np.isfinite(endog.iloc[position - 1]) or not np.isfinite(actual).all():
            continue
        positions.append(position)
    return np.asarray(positions, dtype=np.int64)


def causal_future_exog(exog, origin_position, known_ahead_steps):
    future = exog.iloc[origin_position : origin_position + FORECAST_STEPS].copy()
    if FUTURE_EXOG_POLICY == "observed_hindcast":
        return future
    if FUTURE_EXOG_POLICY != "causal_persistence":
        raise ValueError("Unknown FUTURE_EXOG_POLICY.")

    previous = exog.iloc[origin_position - 1]
    for column in future:
        if column in KNOWN_FUTURE_COLUMNS:
            continue
        known = min(int(known_ahead_steps.get(column, 0)), FORECAST_STEPS)
        if known >= FORECAST_STEPS:
            continue
        fill_value = previous[column] if known == 0 else future.iloc[known - 1][column]
        future.iloc[known:, future.columns.get_loc(column)] = fill_value
    return future


def make_model(design, order, seasonal_order):
    train = period_mask(design["endog"].index, "train")
    return SARIMAX(
        design["endog"].loc[train],
        exog=design["exog"].loc[train],
        order=tuple(order),
        seasonal_order=tuple(seasonal_order),
        trend="c",
        enforce_stationarity=False,
        enforce_invertibility=False,
        concentrate_scale=True,
        missing="none",
    )


def design_fingerprint(design, order, seasonal_order):
    train = period_mask(design["endog"].index, "train")
    payload = pd.concat(
        [design["endog"].loc[train].rename("endog"), design["exog"].loc[train]], axis=1
    )
    digest = hashlib.sha256()
    digest.update(pd.util.hash_pandas_object(payload, index=True).to_numpy().tobytes())
    digest.update(json.dumps({
        "lag": design["terneuzen_lag_hours"],
        "order": list(order),
        "seasonal_order": list(seasonal_order),
        "columns": list(design["exog"].columns),
    }, sort_keys=True).encode("utf-8"))
    return digest.hexdigest()


def cache_path(design, order, seasonal_order):
    p, d, q = order
    P, D, Q, m = seasonal_order
    return CACHE_DIR / (
        f"lag{design['terneuzen_lag_hours']}_p{p}d{d}q{q}_P{P}D{D}Q{Q}m{m}.npz"
    )


def fit_or_load(design, order, seasonal_order):
    model = make_model(design, order, seasonal_order)
    fingerprint = design_fingerprint(design, order, seasonal_order)
    path = cache_path(design, order, seasonal_order)
    if path.exists() and not FORCE_REFIT:
        with np.load(path, allow_pickle=False) as saved:
            if saved["fingerprint"].item() == fingerprint:
                params = saved["params"]
                started = time.perf_counter()
                fitted = model.filter(params)
                converged = bool(saved["converged"].item()) if "converged" in saved else False
                return fitted, {"Fit seconds": time.perf_counter() - started, "Loaded cache": True, "Converged": converged}

    started = time.perf_counter()
    fitted = model.fit(method="lbfgs", maxiter=MAXITER, disp=False)
    elapsed = time.perf_counter() - started
    converged = bool(fitted.mle_retvals.get("converged", False))
    np.savez_compressed(
        path,
        params=np.asarray(fitted.params, dtype=np.float64),
        fingerprint=np.array(fingerprint),
        converged=np.array(converged),
    )
    return fitted, {"Fit seconds": elapsed, "Loaded cache": False, "Converged": converged}


def rolling_forecast(fitted, design, split_name, stride_steps, *, progress=False):
    endog, exog = design["endog"], design["exog"]
    origins = eligible_origin_positions(endog, split_name, stride_steps)
    if not len(origins):
        raise RuntimeError(f"No complete {split_name} forecast origins.")

    train_positions = np.flatnonzero(period_mask(endog.index, "train"))
    state_end = int(train_positions[-1])
    state = fitted
    actual_rows, mean_rows, lower_rows, upper_rows, baselines, origin_times = [], [], [], [], [], []

    for number, origin in enumerate(origins, start=1):
        if origin > state_end + 1:
            update = slice(state_end + 1, origin)
            state = state.extend(endog.iloc[update], exog=exog.iloc[update])
            state_end = origin - 1

        future_exog = causal_future_exog(exog, origin, design["known_ahead_steps"])
        forecast = state.get_forecast(steps=FORECAST_STEPS, exog=future_exog)
        interval = forecast.conf_int(alpha=INTERVAL_ALPHA)

        actual_rows.append(endog.iloc[origin : origin + FORECAST_STEPS].to_numpy(dtype=np.float64))
        mean_rows.append(np.asarray(forecast.predicted_mean, dtype=np.float64))
        lower_rows.append(interval.iloc[:, 0].to_numpy(dtype=np.float64))
        upper_rows.append(interval.iloc[:, 1].to_numpy(dtype=np.float64))
        baselines.append(float(endog.iloc[origin - 1]))
        origin_times.append(endog.index[origin])

        if progress and (number == 1 or number % 50 == 0 or number == len(origins)):
            print(f"  {split_name}: {number}/{len(origins)} origins", flush=True)

    return {
        "actual": np.vstack(actual_rows),
        "prediction": np.vstack(mean_rows),
        "lower": np.vstack(lower_rows),
        "upper": np.vstack(upper_rows),
        "baseline": np.asarray(baselines, dtype=np.float64),
        "origins": pd.DatetimeIndex(origin_times),
    }


def aggregate_metrics(bundle, mask=None):
    actual = bundle["actual"] if mask is None else bundle["actual"][mask]
    predicted = bundle["prediction"] if mask is None else bundle["prediction"][mask]
    baseline_values = bundle["baseline"] if mask is None else bundle["baseline"][mask]
    persistence = np.repeat(baseline_values[:, None], FORECAST_STEPS, axis=1)
    lower = bundle["lower"] if mask is None else bundle["lower"][mask]
    upper = bundle["upper"] if mask is None else bundle["upper"][mask]
    model_rmse = float(np.sqrt(mean_squared_error(actual.ravel(), predicted.ravel())))
    persistence_rmse = float(np.sqrt(mean_squared_error(actual.ravel(), persistence.ravel())))
    return {
        "MAE": float(mean_absolute_error(actual.ravel(), predicted.ravel())),
        "RMSE": model_rmse,
        "R2": float(r2_score(actual.ravel(), predicted.ravel())),
        "Persistence RMSE": persistence_rmse,
        "RMSE skill": 1.0 - model_rmse / persistence_rmse,
        "Interval coverage": float(((actual >= lower) & (actual <= upper)).mean()),
        "Mean interval width": float((upper - lower).mean()),
        "Origins": len(actual),
    }


## 4. Select the order on August only

Each candidate is estimated on January–July. Its fixed parameters are then updated through August observations with the state-space `extend` operation **without refitting**, and rolling 72-hour forecasts are scored at six-hour origin spacing. Candidate parameters are checkpointed under `time_series_analysis/model_cache/sarimax`, so an interrupted search can resume without repeating successful optimizations.


In [ ]:
search_rows = []

for candidate_number, (order, seasonal_order) in enumerate(CANDIDATES, start=1):
    print(f"[{candidate_number}/{len(CANDIDATES)}] order={order}, seasonal_order={seasonal_order}", flush=True)
    try:
        fitted, fit_info = fit_or_load(baseline_design, order, seasonal_order)
        validation_bundle = rolling_forecast(
            fitted,
            baseline_design,
            "validation",
            VALIDATION_ORIGIN_STRIDE_STEPS,
        )
        score = aggregate_metrics(validation_bundle)
        search_rows.append({
            "order": str(order),
            "seasonal_order": str(seasonal_order),
            "Terneuzen lag hours": BASELINE_TERNEUZEN_LAG_HOURS,
            "AIC (diagnostic only)": float(fitted.aic),
            "Error": "",
            **fit_info,
            **{f"Validation {name}": value for name, value in score.items()},
        })
        print(f"  August RMSE={score['RMSE']:.5f}; skill={score['RMSE skill']:.3f}")
        del validation_bundle, fitted
    except Exception as error:
        search_rows.append({
            "order": str(order),
            "seasonal_order": str(seasonal_order),
            "Terneuzen lag hours": BASELINE_TERNEUZEN_LAG_HOURS,
            "Error": f"{type(error).__name__}: {error}",
        })
        print("  FAILED:", search_rows[-1]["Error"])
    pd.DataFrame(search_rows).to_csv(RESULTS_DIR / "order_search_progress.csv", index=False)
    gc.collect()

order_search = pd.DataFrame(search_rows)
successful_orders = order_search[
    order_search["Error"].fillna("").eq("") & order_search["Validation RMSE"].notna()
].sort_values("Validation RMSE")
if successful_orders.empty:
    raise RuntimeError("Every SARIMAX candidate failed. Inspect order_search_progress.csv.")

display(successful_orders.round({
    "Fit seconds": 1,
    "AIC (diagnostic only)": 1,
    "Validation MAE": 5,
    "Validation RMSE": 5,
    "Validation R2": 3,
    "Validation RMSE skill": 3,
    "Validation Interval coverage": 3,
    "Validation Mean interval width": 4,
}))


## 5. Test the 42-hour hypothesis on August

The best order from the baseline 42-hour design is held fixed. The model is then estimated separately with 36-, 42-, and 48-hour Terneuzen designs and again ranked only on August rolling RMSE. This small sensitivity analysis prevents the existing 42-hour feature from being treated as settled physics while avoiding a full order-by-lag Cartesian search.


In [ ]:
best_order = tuple(int(value) for value in successful_orders.iloc[0]["order"].strip("()").split(","))
best_seasonal_order = tuple(
    int(value) for value in successful_orders.iloc[0]["seasonal_order"].strip("()").split(",")
)

lag_rows = []
for lag_hours in TERNEUZEN_LAG_HYPOTHESES:
    print(f"Evaluating Terneuzen lag {lag_hours} h", flush=True)
    design = build_design(lag_hours)
    try:
        fitted, fit_info = fit_or_load(design, best_order, best_seasonal_order)
        bundle = rolling_forecast(
            fitted,
            design,
            "validation",
            VALIDATION_ORIGIN_STRIDE_STEPS,
        )
        score = aggregate_metrics(bundle)
        lag_rows.append({
            "Terneuzen lag hours": lag_hours,
            "order": str(best_order),
            "seasonal_order": str(best_seasonal_order),
            "Error": "",
            **fit_info,
            **{f"Validation {name}": value for name, value in score.items()},
        })
        del fitted, bundle
    except Exception as error:
        lag_rows.append({
            "Terneuzen lag hours": lag_hours,
            "order": str(best_order),
            "seasonal_order": str(best_seasonal_order),
            "Error": f"{type(error).__name__}: {error}",
        })
    pd.DataFrame(lag_rows).to_csv(RESULTS_DIR / "terneuzen_lag_search_progress.csv", index=False)
    gc.collect()

lag_search = pd.DataFrame(lag_rows)
successful_lags = lag_search[
    lag_search["Error"].fillna("").eq("") & lag_search["Validation RMSE"].notna()
].sort_values("Validation RMSE")
if successful_lags.empty:
    raise RuntimeError("Every Terneuzen lag hypothesis failed.")

selected_lag_hours = int(successful_lags.iloc[0]["Terneuzen lag hours"])
selection = {
    "order": best_order,
    "seasonal_order": best_seasonal_order,
    "Terneuzen lag hours": selected_lag_hours,
    "selected by": "minimum August rolling 72-hour RMSE",
    "future exog policy": FUTURE_EXOG_POLICY,
}
(RESULTS_DIR / "selected_configuration.json").write_text(
    json.dumps(selection, indent=2), encoding="utf-8"
)
display(successful_lags.round({
    "Fit seconds": 1,
    "Validation MAE": 5,
    "Validation RMSE": 5,
    "Validation R2": 3,
    "Validation RMSE skill": 3,
    "Validation Interval coverage": 3,
}))
print("Selected configuration:", selection)


## 6. Held-out September evaluation

Only now is September opened. The selected January–July parameter estimates are filtered through the observed August sequence without re-estimating them, then evaluated with hourly rolling forecast origins. Set `TEST_ORIGIN_STRIDE_STEPS=1` for every 15-minute origin; the default hourly spacing is a runtime-conscious compromise while preserving the exact September test period.

Reported outputs include MAE, RMSE, R², skill versus persistence, performance at each forecast lead, the largest 10% of conductivity-change events, and empirical coverage/width of the nominal 90% conditional intervals.


In [ ]:
selected_design = build_design(selected_lag_hours)
selected_fitted, selected_fit_info = fit_or_load(
    selected_design, best_order, best_seasonal_order
)
test_bundle = rolling_forecast(
    selected_fitted,
    selected_design,
    "test",
    TEST_ORIGIN_STRIDE_STEPS,
    progress=True,
)

event_size = np.max(
    np.abs(test_bundle["actual"] - test_bundle["baseline"][:, None]), axis=1
)
large_change_mask = event_size >= np.quantile(event_size, 0.90)

summary_rows = [
    {"Scope": "all horizons", **aggregate_metrics(test_bundle)},
    {"Scope": "largest 10% changes", **aggregate_metrics(test_bundle, large_change_mask)},
]

lead_rows = []
for step in range(FORECAST_STEPS):
    actual = test_bundle["actual"][:, step]
    prediction = test_bundle["prediction"][:, step]
    persistence = test_bundle["baseline"]
    lower = test_bundle["lower"][:, step]
    upper = test_bundle["upper"][:, step]
    model_rmse = float(np.sqrt(mean_squared_error(actual, prediction)))
    persistence_rmse = float(np.sqrt(mean_squared_error(actual, persistence)))
    lead_rows.append({
        "Lead minutes": (step + 1) * RESAMPLE_MINUTES,
        "Lead hours": (step + 1) / STEPS_PER_HOUR,
        "MAE": float(mean_absolute_error(actual, prediction)),
        "RMSE": model_rmse,
        "R2": float(r2_score(actual, prediction)),
        "Persistence RMSE": persistence_rmse,
        "RMSE skill": 1.0 - model_rmse / persistence_rmse,
        "Interval coverage": float(((actual >= lower) & (actual <= upper)).mean()),
        "Mean interval width": float((upper - lower).mean()),
    })

lead_results = pd.DataFrame(lead_rows)
for hours in (*EVALUATION_HOURS, FORECAST_HOURS):
    row = lead_results.iloc[hours * STEPS_PER_HOUR - 1].to_dict()
    summary_rows.append({"Scope": f"{hours}-hour mark", "Origins": len(test_bundle["origins"]), **row})

test_summary = pd.DataFrame(summary_rows)
test_summary.to_csv(RESULTS_DIR / "september_summary.csv", index=False)
lead_results.to_csv(RESULTS_DIR / "september_by_forecast_lead.csv", index=False)
np.savez_compressed(
    RESULTS_DIR / "september_forecasts.npz",
    origins=test_bundle["origins"].to_numpy(dtype="datetime64[ns]"),
    actual=test_bundle["actual"],
    prediction=test_bundle["prediction"],
    lower=test_bundle["lower"],
    upper=test_bundle["upper"],
    baseline=test_bundle["baseline"],
)

display(test_summary.round({
    "MAE": 5,
    "RMSE": 5,
    "R2": 3,
    "Persistence RMSE": 5,
    "RMSE skill": 3,
    "Interval coverage": 3,
    "Mean interval width": 4,
}))
print("Nominal interval coverage:", 1.0 - INTERVAL_ALPHA)
print("Results saved to:", RESULTS_DIR)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

axes[0].plot(lead_results["Lead hours"], lead_results["RMSE"], label="SARIMAX")
axes[0].plot(lead_results["Lead hours"], lead_results["Persistence RMSE"], label="Persistence", linestyle="--")
axes[0].set(title="September error by forecast lead", xlabel="Lead hours", ylabel="RMSE")
axes[0].legend()

axes[1].plot(lead_results["Lead hours"], lead_results["RMSE skill"], color="tab:green")
axes[1].axhline(0.0, color="black", linewidth=1)
axes[1].set(title="Skill versus persistence", xlabel="Lead hours", ylabel="RMSE skill")

axes[2].plot(lead_results["Lead hours"], lead_results["Interval coverage"], label="Empirical")
axes[2].axhline(1.0 - INTERVAL_ALPHA, color="black", linestyle="--", label="Nominal")
axes[2].set_ylim(0, 1)
axes[2].set(title="Conditional prediction-interval coverage", xlabel="Lead hours", ylabel="Coverage")
axes[2].legend()

fig.tight_layout()
plt.show()

# Illustrate one forecast origin chosen without looking for a visually favorable case.
example = len(test_bundle["origins"]) // 2
times = pd.date_range(
    test_bundle["origins"][example], periods=FORECAST_STEPS, freq=f"{RESAMPLE_MINUTES}min"
)
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(times, test_bundle["actual"][example], label="Actual", color="black")
ax.plot(times, test_bundle["prediction"][example], label="SARIMAX", color="tab:blue")
ax.plot(times, np.repeat(test_bundle["baseline"][example], FORECAST_STEPS), label="Persistence", linestyle="--")
ax.fill_between(
    times,
    test_bundle["lower"][example],
    test_bundle["upper"][example],
    alpha=0.2,
    color="tab:blue",
    label=f"{int((1 - INTERVAL_ALPHA) * 100)}% conditional interval",
)
ax.set(title=f"Mid-September evaluation origin: {test_bundle['origins'][example]}", ylabel="Conductivity (mS/cm)")
ax.legend(ncol=4)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()


## 7. Interpretation and reporting checklist

Before drawing conclusions, report all of the following:

1. The selected `(p,d,q)` and `(P,D,Q,96)` and whether optimization converged.
2. The complete August order table, not only the winner.
3. The 36/42/48-hour validation comparison. A 42-hour win supports the hypothesis within this small candidate set; it does not establish a universal propagation time.
4. Which environmental variables were actually loaded and their coverage.
5. The future-exogenous policy. Results from `observed_hindcast` must be labelled oracle/hindcast results and must not be presented as deployable forecasts.
6. September aggregate, large-change, checkpoint, and per-lead metrics against persistence.
7. Nominal versus empirical interval coverage and interval width. SARIMAX intervals are conditional on the model, fixed parameters, and supplied exogenous path; they do not include uncertainty in future rain, discharge, traffic, or other exogenous forecasts.
8. The test-origin stride. Use stride 1 if exact every-15-minute comparability with the existing neural/classical evaluation is required.

The strongest comparison is not whether SARIMAX has the best AIC, but whether its August-selected configuration improves held-out September skill while retaining useful coverage during the largest conductivity changes.
